In [1]:
import os
import sys
import re 
import itertools

# from datasets import load_dataset
from tqdm import tqdm
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  i

In [2]:
seed = 888

data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis/'
author = 'Tahoe100M'

In [3]:
# drug_ds = load_dataset("tahoebio/Tahoe-100M", "drug_metadata", split="train")
# drug_ds = drug_ds.to_pandas()

url = "https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/drug_metadata.parquet"
drug_ds = pd.read_parquet(url)

exclude_drugs = drug_ds[drug_ds.targets.isna()].drug.unique().tolist()
keep_drugs = drug_ds[~drug_ds.targets.isna()].drug.unique().tolist()
# keep_drugs.append('DMSO_TF')
print('{} of {} are excluded because they have unknown targets'.format(len(exclude_drugs), drug_ds.shape[0]))
drug_ds = drug_ds[drug_ds.drug.isin(keep_drugs)]

115 of 379 are excluded because they have unknown targets


Get the ominpath nodes:

In [4]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

sn_ppis = parse_network.load_network('omnipath', organism = 'human', static = True)
sn_ppis = parse_network.correct_network(sn_ppis = sn_ppis,
                                        source_label = source_label, target_label = target_label,
                                        stimulation_label = stimulation_label, inhibition_label = inhibition_label)
sn_ppis = parse_network.extract_network(sn_ppis = sn_ppis.copy(), 
                                        curation_effort_thresh = 2, 
                          n_references_thresh = 1,
                          resources = 'all', 
                          drop_self = True, 
                          source_label = source_label, 
                          target_label = target_label,
                          verbose = False)

ppi_nodes = sn_ppis.source_genesymbol.tolist() + sn_ppis.target_genesymbol.tolist()

# also get the collectri tfs
grn_link = 'https://zenodo.org/records/11477837/files/collectri_human_06_04_24.csv'
tf_net = pd.read_csv(grn_link, index_col = 0)


/home/hmbaghda/Projects/scLEMBAS/scLEMBAS/parse_network.py:162: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unique_vals = pd.concat([unique_vals, dup_int], axis = 0)


Exclude drugs that do not target Omnipath nodes or that target CollecTri TFs:

In [5]:
# check whether drug targets are present in omnipath
drug_targets = {item for sublist in drug_ds.targets.apply(lambda x: re.split(r',\s*', x)) for item in sublist}
missing_targets = drug_targets.difference(ppi_nodes)

def check_omnipath_present(x, missing_targets):
    """Checks whether drug has atleast one target present in omnipath (False) or not (True)"""
    target_list = set(re.split(r',\s*', x))
    return len(target_list.difference(missing_targets)) == 0
    
missing_drugs = set(drug_ds[drug_ds.targets.apply(lambda x: check_omnipath_present(x, 
                                                                          missing_targets=missing_targets))].drug)


def check_tf_target(x, tfs):
    """Checks whether the drug only targets a TF (true)."""
    target_list = set(re.split(r',\s*', x))
    b = len(target_list) == 1
    a = len(target_list.intersection(tf_net.source)) == 1
    return a and b

tf_targetting_drugs = set(drug_ds[drug_ds.targets.apply(lambda x: check_tf_target(x, tf_net.source))].drug)

drop_drugs = tf_targetting_drugs.union(missing_drugs)

msg = '{} drugs do not have a target present in Omnipath'.format(len(missing_drugs))
msg += ', and {} drugs target only a TF directly'.format(len(tf_targetting_drugs))
msg += '. Overall, we exclude {} drugs for this'.format(len(drop_drugs))
print(msg)

drug_ds = drug_ds[~drug_ds.drug.isin(drop_drugs)]
keep_drugs = set(drug_ds.drug)

15 drugs do not have a target present in Omnipath, and 25 drugs target only a TF directly. Overall, we exclude 40 drugs for this


Finally, let's only include drugs with known MOAs:

In [6]:
# this is simply to further filter n cells for compute time
known_moa = set(drug_ds[(drug_ds['moa-fine'] != 'unclear') & 
       (drug_ds['moa-broad'] != 'unclear')].drug)
print('An additional {} with unknown MOAs are dropped'.format(len(keep_drugs.difference(known_moa))))

drug_ds = drug_ds[drug_ds.drug.isin(known_moa)]
keep_drugs = set(drug_ds.drug)

An additional 96 with unknown MOAs are dropped


Get the drug doses per sample. We will only retain the highest dose for our analyses.

In [7]:
url = 'https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/sample_metadata.parquet'
sample_md = pd.read_parquet(url)
sample_md.drug = sample_md.drug.str.rstrip(' ')

sample_md['dose (uM)'] = sample_md.drugname_drugconc.apply(lambda x: eval(x)[0][1])

# check for which drug-dose combinations are present (all are present)
all_drugs = sample_md['drug'].dropna().unique()
all_doses = sample_md['dose (uM)'].dropna().unique()
# drop control
all_drugs = all_drugs[all_drugs != 'DMSO_TF']
all_doses = all_doses[all_doses != 0] 

all_combinations = pd.DataFrame(list(itertools.product(all_drugs, all_doses)), columns=['drug', 'dose (uM)'])
observed = sample_md[['drug', 'dose (uM)']].drop_duplicates()

missing = pd.merge(all_combinations, observed, on=['drug', 'dose (uM)'], how='left', indicator=True)
missing_combinations = missing[missing['_merge'] == 'left_only'].drop(columns=['_merge'])
if missing_combinations.shape[0] != 0:
    raise ValueError('Unexpected missing drug-dose combination')
    
keep_samples = sample_md[sample_md['dose (uM)'].isin([5, 0])]['sample'].tolist()
print('{} of {} samples are retained  at the highest dosage (or control)'.format(len(keep_samples), sample_md.shape[0]))


keep_drugs.add('DMSO_TF')
sample_md = sample_md[sample_md['sample'].isin(keep_samples) & sample_md.drug.isin(keep_drugs)]
print('{} of the retained samples overlap with the drugs with known targets'.format(sample_md.shape[0]))

dropped_drugs = set(keep_drugs).difference(sample_md.drug)
if len(dropped_drugs) > 0:
    raise ValueError('Unexpected filtering of drugs here')
    print('With these sample filters, an additional {} drugs are dropped'.format(len(dropped_drugs)))
    keep_drugs = set(keep_drugs).intersection(sample_md.drug)

keep_samples = sample_md['sample'].tolist()


518 of 1344 samples are retained  at the highest dosage (or control)
184 of the retained samples overlap with the drugs with known targets


In [8]:
url = 'https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/obs_metadata.parquet'
obs = pd.read_parquet(url)
obs.drug = obs.drug.str.rstrip(' ')
backup = obs.copy()

n_total = obs.shape[0]
obs = obs[obs['sample'].isin(keep_samples)]
print('{} of {} total measured cells are retained after the above filters'.format(obs.shape[0], 
                                                                                 n_total))

13544750 of 100648790 total measured cells are retained after the above filters


Next, we only include the cells that pass full QC filters:

In [9]:
n_cells_tot = obs.shape[0]
obs = obs[obs.pass_filter == 'full']

msg = '{} of {} cells pass full QC filters'.format(obs.shape[0], n_cells_tot)

dropped_samples = set(keep_samples).difference(obs['sample'])
dropped_drugs = set(keep_drugs).difference(obs['drug'])
if len(dropped_drugs) > 0 or len(dropped_samples) > 0:
    msg += ', resulting in an additional {} samples and {} drugs being dropped'.format(len(dropped_samples), 
                                                                           len(dropped_drugs))
    
    keep_drugs = set(keep_drugs).intersection(obs.drug)
    keep_samples = set(keep_samples).intersection(obs['sample'])

print(msg)

12788558 of 13544750 cells pass full QC filters


Next, we only include conditions (cell line - drug combinations) with atleast n_cells_per_cond cells (we run this excluding control cells):

In [10]:
ctrl_cells = obs[obs.drug == 'DMSO_TF']
obs = obs[obs.drug != 'DMSO_TF']

In [11]:
n_cells_per_cond = 1750

obs['condition'] = obs.cell_line.astype(str) + '^' + obs.drug.astype(str)
condition_counts = obs['condition'].value_counts()

n_start_conds = condition_counts.nunique()
keep_conds = condition_counts[condition_counts >= n_cells_per_cond]

msg = 'Excluding conditions with fewer than {} cells decreases the total conditions from {} to {}'.format(n_cells_per_cond, 
                                                                                                          condition_counts.index.nunique(), 
                                                                                                          keep_conds.index.nunique())

obs = obs[obs['condition'].isin(keep_conds.index)]
msg += ', decreasing the total number of cells to {}'.format(obs.shape[0])

dropped_samples = set(keep_samples).difference(obs['sample'])
dropped_drugs = set(keep_drugs).difference(obs['drug'])
if len(dropped_drugs) > 0 or len(dropped_samples) > 0:
    msg += ' and resulting in an additional {} samples and {} drugs being dropped'.format(len(dropped_samples), 
                                                                           len(dropped_drugs))
    
    keep_drugs = set(keep_drugs).intersection(obs.drug)
    keep_samples = set(keep_samples).intersection(obs['sample'])
    
print(msg)



Excluding conditions with fewer than 1750 cells decreases the total conditions from 6400 to 2354, decreasing the total number of cells to 7546845 and resulting in an additional 49 samples and 22 drugs being dropped


Excluding conditions with fewer than 1750 cells decreases the total conditions from 6400 to 2354, decreasing the total number of cells to 7546845 and resulting in an additional 21 samples and 21 drugs being dropped

In [12]:
url = 'https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/cell_line_metadata.parquet'
cell_line_md = pd.read_parquet(url)

url = "https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/drug_metadata.parquet"
drug_ds = pd.read_parquet(url)

cell_line_md = cell_line_md[cell_line_md['Cell_ID_Cellosaur'].isin(obs.cell_line.unique().tolist())]
drug_ds = drug_ds[drug_ds.drug.isin(obs.drug)]

To ensure good coverage across cell lines/drugs, we only retain drugs present in a majority (75%) of cell lines) and cell lines present in a majority (65%, since there are more drugs) of drugs:

In [13]:
condition_counts = (
    obs.groupby(['drug', 'cell_line']).size().rename('n_cells').reset_index()
)

all_cell_lines = obs['cell_line'].nunique()
all_drugs = obs['drug'].nunique()
presence_matrix = (
    condition_counts
    .assign(present=lambda d: d['n_cells'] > 0)
    .pivot(index='drug', columns='cell_line', values='present')
    .fillna(False)
)


drug_thresh = 0.75
drug_valid = presence_matrix.sum(axis=1) >= (drug_thresh * all_cell_lines)
valid_drugs = presence_matrix.index[drug_valid]

cell_thresh = 0.65
cell_line_valid = presence_matrix.sum(axis=0) >= (cell_thresh * all_drugs)
valid_cell_lines = presence_matrix.columns[cell_line_valid]

obs = obs[obs.cell_line.isin(valid_cell_lines) & obs.drug.isin(valid_drugs)]


########################### print filter results ###########################
msg = '{} drugs are present in {:.2f}% of cell lines and will be retained. '.format(len(valid_drugs), 
                                                                                 100*drug_thresh)
msg += '{} cell lines are present in {:.2f}% of drugs and will be retained. '.format(len(valid_cell_lines), 
                                                                                 100*cell_thresh)
msg += 'This results in {:.3f} million cells and {} conditions'.format(obs.shape[0]/1e6, 
                                                                      obs.condition.nunique())
print(msg)

filtered_cmd = cell_line_md[cell_line_md['Cell_ID_Cellosaur'].isin(obs.cell_line.unique().tolist())]
filtered_dds = drug_ds[drug_ds.drug.isin(obs.drug)]

msg = 'These cell lines represent {} of {} remaining organs'.format(filtered_cmd.Organ.nunique(), 
                                                          cell_line_md.Organ.nunique())
msg += ' and {} of {} remaining driver genes'.format(filtered_cmd.Driver_Gene_Symbol.nunique(), 
                                          cell_line_md.Driver_Gene_Symbol.nunique())
print(msg)

msg = 'These drugs represent {} of {} remaining MOAs '.format(filtered_dds['moa-fine'].nunique(), 
                                                  drug_ds['moa-fine'].nunique())
all_targets = set(itertools.chain.from_iterable(drug_ds.targets.apply(lambda x: list(set(re.split(r',\s*', x)))).tolist()))
filtered_targets = set(itertools.chain.from_iterable(filtered_dds.targets.apply(lambda x: list(set(re.split(r',\s*', x)))).tolist()))

msg += 'and {} of {} remaining targets'.format(len(filtered_targets), 
                                     len(all_targets)
                                    )
print(msg)


/tmp/ipykernel_3620527/1764523604.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs.groupby(['drug', 'cell_line']).size().rename('n_cells').reset_index()


34 drugs are present in 75.00% of cell lines and will be retained. 10 cell lines are present in 65.00% of drugs and will be retained. This results in 1.774 million cells and 340 conditions
These cell lines represent 5 of 12 remaining organs and 39 of 63 remaining driver genes
These drugs represent 13 of 21 remaining MOAs and 64 of 120 remaining targets


Let's add back in the control cells. We'll only add those in the cell lines of interest, and from the remaining plate as we've found that the plates can be a source of variance:

In [14]:
remaining_cell_lines = obs.cell_line.unique()
remaining_plates = obs.plate.unique()

ctrl_cells = ctrl_cells[ctrl_cells.cell_line.isin(remaining_cell_lines) & 
           ctrl_cells.plate.isin(remaining_plates)]

ctrl_cells['condition'] = ctrl_cells.cell_line.astype(str) + '^' + ctrl_cells.drug.astype(str)
ctrl_cells = ctrl_cells[obs.columns]
obs = pd.concat([obs, ctrl_cells], axis = 0)


Finally, to maintain balanced classes, we downsample all conditions to the minimum size:

**Note: When splitting the data into train-test splits, conditions will remain balanced, but drug/cell lines will not be completely balanced (due to OOD, cannot ensure the drug and cell line split is the same as the condition split).** 

In [15]:
obs.reset_index(drop = True, inplace = True)

condition_counts = (
    obs.groupby(['drug', 'cell_line'], observed = False).size().rename('n_cells').reset_index()
)
condition_counts = condition_counts[condition_counts.n_cells != 0]
min_cond = condition_counts.n_cells.min()


print('The minimum number of cells in a given condition is {}'.format(min_cond))

The minimum number of cells in a given condition is 2099


In [16]:
# quantile_cutoff = 0.25
# target_cells_per_drug = (
#     condition_counts.groupby('drug')['n_cells']
#     .quantile(quantile_cutoff)
#     .astype(int)
#     .rename('target_cells')
# )
# condition_counts = condition_counts.merge(target_cells_per_drug, on='drug')

condition_counts['target_cells'] = min_cond

In [17]:
grouped = obs.groupby('condition')

downsampled_subsets = []
for _, row in tqdm(condition_counts.iterrows()):
    condition = row['cell_line'] + '^' + row['drug']
    subset_df = grouped.get_group(condition)
    if row.n_cells > row.target_cells:
        subset_df = subset_df.sample(n = row.target_cells,
                             random_state = seed,
                             replace = False)
    downsampled_subsets.append(subset_df)
obs = pd.concat(downsampled_subsets, ignore_index=True)
for col in obs.select_dtypes(include='category').columns:
    obs[col] = obs[col].cat.remove_unused_categories()

350it [00:00, 401.58it/s]


In [18]:
n_cl, n_drug, n_cond = obs.cell_line.nunique(), obs.drug.nunique(), obs.condition.nunique()

print('The final {} conditions span {} cell lines and {} drugs (including control)'.format(n_cond, n_cl, n_drug))

The final 350 conditions span 10 cell lines and 35 drugs (including control)


In [19]:
obs.cell_line.value_counts().unique()

array([73465])

In [20]:
obs.drug.value_counts().unique()

array([20990])

In [21]:
obs.shape[0]

734650

Add the drug targets to the single-cell metadata:

In [22]:
url = "https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/drug_metadata.parquet"
drug_ds = pd.read_parquet(url)

if drug_ds.drug.nunique() != drug_ds.shape[0]:
    raise ValueError('Expected only unique drugs')
drug_ds = drug_ds[drug_ds.targets.notna()]

drug_target_map = dict(zip(
    drug_ds.drug, 
    drug_ds.targets.apply(lambda x: set(re.split(r',\s*', x)))
))
drug_target_map['DMSO_TF'] = {}

drug_moa = dict(zip(drug_ds.drug, drug_ds['moa-broad']))
drug_moa['DMSO_TF'] = None

obs['drug_target'] = obs.drug.map(drug_target_map)
obs['drug_moa'] = obs.drug.map(drug_moa)

In [25]:
if obs.BARCODE_SUB_LIB_ID.nunique() != obs.shape[0]:
    raise ValueError('Expected unique barcode per cell')
else:
    obs.set_index('BARCODE_SUB_LIB_ID', inplace = True)
    
# obs_final = obs.copy()
obs.to_csv(os.path.join(data_path, 'processed', author + '_filtered_cell_metadata.csv'))